<a href="https://colab.research.google.com/github/sumaiyanawaz776-code/Machine-Learning-Advance/blob/main/News_Classification_with_BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
!pip install -q transformers datasets evaluate accelerate gradio

In [22]:
import torch

In [23]:
print("GPU Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

GPU Available: True
GPU Name: Tesla T4


In [24]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)

import gradio as gr
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

In [25]:
dataset = load_dataset("ag_news")
print(dataset)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})
{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


In [26]:
train_data = dataset["train"].shuffle(seed=42).select(range(8000))
test_data = dataset["test"].shuffle(seed=42).select(range(2000))

In [27]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(batch):
  return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

train_data = train_data.map(tokenize_function, batched=True)
test_data = test_data.map(tokenize_function, batched=True)


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [28]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4,
    id2label={0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"},
    label2id={"World": 0, "Sports": 1, "Business": 2, "Sci/Tech": 3}
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [29]:
!pip install -q evaluate

In [30]:
import evaluate

In [31]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none",
    fp16=True,
    dataloader_num_workers=2
)


accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis=-1)
  acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
  f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")["f1"]
  return {"accuracy": acc, "f1_score": f1}

In [32]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Score
1,0.397199,0.303533,0.904000,0.903668
2,0.200425,0.298537,0.914500,0.914448
3,0.117745,0.325239,0.913000,0.912987


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=1500, training_loss=0.23845631663004557, metrics={'train_runtime': 224.4734, 'train_samples_per_second': 106.917, 'train_steps_per_second': 6.682, 'total_flos': 1578694680576000.0, 'train_loss': 0.23845631663004557, 'epoch': 3.0})

In [33]:
results = trainer.evaluate()
print(results)

{'eval_loss': 0.2984329164028168, 'eval_accuracy': 0.9145, 'eval_f1_score': 0.9144484416923025, 'eval_runtime': 3.8715, 'eval_samples_per_second': 516.601, 'eval_steps_per_second': 32.288, 'epoch': 3.0}


In [34]:
predictions = trainer.predict(test_data)
preds = np.argmax(predictions.predictions, axis=-1)
print("Test Accuracy:", accuracy_metric.compute(predictions=preds, references=test_data["label"])["accuracy"])
print("Test F1 Score:", f1_metric.compute(predictions=preds, references=test_data["label"], average="weighted")["f1"])

Test Accuracy: 0.9145
Test F1 Score: 0.9144484416923025


In [35]:
trainer.save_model("./result")
tokenizer.save_pretrained("./result")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./result/tokenizer_config.json', './result/tokenizer.json')

In [36]:
classifier = pipeline(
    "text-classification",
    model="./bert-ag-news",
    tokenizer="./bert-ag-news",
    device=0 if torch.cuda.is_available() else -1
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [37]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("./bert-ag-news")
id2label = config.id2label

def predict_news(text):
  result = classifier(text, truncation=True, max_length=128)[0]
  label = result["label"]
  return f"Predicted Label:  {label}"

In [38]:
demo = gr.Interface(
    fn=predict_news,
    inputs=gr.Textbox(lines=5, placeholder="Enter news text here..."),
    outputs=gr.Markdown(),
    title="News Classification with BERT",
    description="Enter a news article and get its classification",
    examples=[
        ["Apple unveils new AI-powered MacBook with M4 chip"],
        ["Real Madrid defeats Barcelona in El Clasico thriller"],
        ["Fed raises interest rates amid inflation concerns"],
        ["NASA confirms discovery of potential signs of life on Mars"],
        ["World leaders gather for climate summit in Geneva"]
    ],
    allow_flagging="never",
    theme="soft"
)

demo.launch(
    share=True,
    debug=True,
    show_error=True
    )

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://21a4601cb783a2c671.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://21a4601cb783a2c671.gradio.live
